In [ ]:
# please start server first with: ./tasks_scripts/ppo/local_m8_680m_server.sh
paths = ["/data03/home/liuxin.ai/alpha_seed", "/data03/home/liuxin.ai/seed_models", "/data03/home/liuxin.ai/verl"]

# paths = ["/opt/tiger/alpha_seed", "/opt/tiger/seed_models", "/opt/tiger/verl"]

import ray
runtime_env = {
    'env_vars': {
        "PYTHONPATH": ":".join(paths)
    }
}

# attach to the server
ray.init(namespace="alphaseed", runtime_env=runtime_env, address="auto")

In [ ]:
import sys
sys.path.extend(paths)
from alpha_seed.workers.actors.async_actor_ref_worker import AsyncActorRolloutRefWorker
from mono_rl.single_controller.ray import RayClassWithInitArgs, RayWorkerGroup
from mono_rl.single_controller.ray import create_colocated_worker_cls

kv_store = ray.get_actor(name="kv_store")
print(kv_store)
worker_names = ray.get(kv_store.get_by_key.remote("hybrid_pool"))
print(worker_names)

actor_rollout_cls = RayClassWithInitArgs(cls=AsyncActorRolloutRefWorker, config=None, role="actor_rollout_ref")
worker_dict_cls = create_colocated_worker_cls(class_dict={"actor_rollout_ref": actor_rollout_cls})

wg_dict = RayWorkerGroup(ray_cls_with_init=worker_dict_cls, worker_names=worker_names)
print(wg_dict)

spawn_wg = wg_dict.spawn(prefix_set=["actor_rollout_ref"])
print(spawn_wg)
wg = spawn_wg["actor_rollout_ref"]
print(wg)
print(wg.workers)

In [ ]:
# print(wg.workers)

# def loss_fn(micro_data, full_entropy, log_prob, seqlen):
#     print(micro_data.batch_size)
    
# wg.set_actor_loss_fn(loss_fn)

In [ ]:
import torch
from transformers import AutoTokenizer
from verl.utils.fs import copy_local_path_from_hdfs

remote_config = ray.get(kv_store.get_by_key.remote("config"))
hdfs_path = remote_config.actor_rollout_ref.model.path
print(hdfs_path)
# model_path = copy_local_path_from_hdfs(hdfs_path)
# print(model_path)
# tokenizer = AutoTokenizer.from_pretrained(model_path)
tokenizer = ray.get(kv_store.get_by_key.remote("tokenizer"))
tokenizer.padding_side = "left"
print(tokenizer("hello world", return_tensors="pt"))


from mono_rl import DataProto
from alpha_seed.utils.dataset.rl_dataset import RLHFDataset, collate_fn
from torch.utils.data import DataLoader


# TRAIN_FILE="hdfs://haruna/home/byte_data_seed/lf_lq/user/zhangchi.usc1992/data/rlhf/math/train_with_ref_ans.parquet"
TRAIN_FILE="hdfs://haruna/home/byte_data_seed/ssd_wlcb/user/qiying/projects/alphaseed/datasets/opensource/deepscaler_train_40k.parquet"
max_prompt_length = 2048
train_dataset = RLHFDataset(parquet_files=TRAIN_FILE,
                            tokenizer=tokenizer,
                            prompt_key="prompt",
                            answer_key="answer",
                            max_prompt_length=max_prompt_length)
print(train_dataset)
print(len(train_dataset))
print(train_dataset[0])

train_dataloader = DataLoader(dataset=train_dataset,
                              batch_size=8,
                              shuffle=None,
                              drop_last=True,
                              collate_fn=collate_fn)
generation_kwargs = {
    'do_sample': True,
    'top_k': 0,
    'top_p': 1.,
    'temperature': 1.,
}
num_bon = 8


# todo: fixme
def set_default_values(batch, max_response_length):
    missing_keys = ["rollout_behavior_log_probs", "probs_gt_threshold_num", "probs_lt_threshold_sum", "off_policy_steps"]
    for key in missing_keys:
        if key not in batch:
            batch.batch[key] = torch.zeros(batch.batch['input_ids'].shape[0],
                                                       remote_config.data.max_response_length,
                                                       dtype=torch.bfloat16,
                                                       device=batch.batch['input_ids'].device).fill_(-1)
    return batch

for batch_dict in train_dataloader:
    batch: DataProto = DataProto.from_single_dict(batch_dict, meta_info={'generation_kwargs': generation_kwargs})
    # print(batch_dict)
    print(batch)

    batch = set_default_values(batch, remote_config.data.max_response_length)
    prompts = tokenizer.batch_decode(batch.batch["input_ids"], skip_special_tokens=True)

    for p in prompts:
        print(p)
        break

    # repeat bon for grpo
    batch = batch.repeat(num_bon)
    # gen !
    gen_out = wg.generate_sequences(batch)
    print(gen_out.batch)

    # log prob !
    gen_out.batch["prompts"] = gen_out.batch["input_ids"][:, :max_prompt_length]
    gen_out.batch["responses"] = gen_out.batch["input_ids"][:, max_prompt_length:]
    gen_out.meta_info = {'generation_kwargs': generation_kwargs}
    gen_out = wg.old_log_probs(gen_out)
    break

In [ ]:
# compute reward
from alpha_seed.utils.reward_score import math_v1

print(gen_out.batch)
batch_size = gen_out.batch.batch_size[0]
sequences = tokenizer.batch_decode(gen_out.batch["input_ids"], skip_special_tokens=True)
print(batch_size)
max_prompt_length = 2048

response_ids = gen_out.batch['input_ids'][:, max_prompt_length:]
reward_tensor = torch.zeros_like(response_ids, dtype=torch.float32)
print(response_ids.shape)
print(reward_tensor.shape)

seqlens = []
scores = []
for i in range(batch_size):
    # compute reward !
    ground_truth = batch[i].non_tensor_batch['reward_model']['ground_truth']
    score = math_v1.compute_score(sequences[i], ground_truth)
    scores.append(score)
    if i == 0:
        print("====sequences====")
        print(sequences[i])
        print("====ground_truth====")
        print(ground_truth)
        print("====score====")
        print(score)

    prompt_length = batch[i].batch['input_ids'].shape[-1]
    valid_response_length = gen_out[i].batch['attention_mask'][prompt_length:].sum().item()
    # print(prompt_length, valid_response_length)
    seqlens.append(valid_response_length)
    
    # give score to the eos token
    reward_tensor[i, valid_response_length - 1] = score
    
print(seqlens)
print(scores)
print(sum(scores) / len(scores))

In [ ]:
# compute grpo adv
from alpha_seed import core_algos

response_length = gen_out.batch['responses'].size(1)
response_mask = gen_out.batch['attention_mask'][:, -response_length:]
index = batch.non_tensor_batch['index']

advantages, returns, adv_metrics = core_algos.compute_grpo_advantage_return(
    token_level_scores=reward_tensor,
    eos_mask=response_mask,
    index=index,
    num_bon=num_bon)

print(reward_tensor)
scores = reward_tensor.sum(-1)
# print(scores)
print(advantages)
print(returns)
# print(adv_metrics)


gen_out.batch['advantages'] = advantages
gen_out.batch['returns'] = returns
gen_out.batch['upgo_advantages'] = torch.zeros_like(advantages)
gen_out.meta_info['global_token_num'] = torch.sum(gen_out.batch['attention_mask'], dim=-1).tolist()

actor_output = wg.update_actor(gen_out)
print(actor_output)

In [ ]:
print(gen_out)

from alpha_seed.workers.ppo_actor import default_loss_fn
from hdfs_io import hcopy
from torch import distributed as dist
print(default_loss_fn)


DEBUG = False

def debug_loss_fn(config, micro_data, full_entropy, log_prob):
    args = {
        "config": config,
        "micro_data": micro_data,
        "full_entropy": full_entropy,
        "log_prob": log_prob,
    }
    if DEBUG and dist.get_rank() == 0:
        torch.save(args, "loss_args.pt")
        print("saving loss_args.pt")
        hcopy("loss_args.pt", remote_config.trainer.default_hdfs_dir)

    loss = default_loss_fn(**args)
    return loss

wg.set_actor_loss_fn(debug_loss_fn)

actor_output = wg.train_actor(gen_out)

In [ ]:
print(remote_config.trainer.default_hdfs_dir)

# hcopy("loss_args.pt", remote_config.trainer.default_hdfs_dir)

args = torch.load("loss_args.pt")
print(args)

import verl.utils.torch_functional as verl_F
def ppo_loss_fn(config, micro_data, full_entropy, log_prob):
    old_log_prob = micro_data['old_log_probs']
    advantages = micro_data['advantages']
    clip_ratio = config.clip_ratio

    responses = micro_data['responses']
    response_length = responses.size(1)
    attention_mask = micro_data['attention_mask']
    response_mask = attention_mask[:, -response_length:]

    ratio = torch.exp(log_prob - old_log_prob)
    pg_losses1 = -advantages * ratio
    pg_losses2 = -advantages * torch.clamp(ratio, 1.0 - clip_ratio, 1.0 + clip_ratio)
    pg_losses_clip = torch.maximum(pg_losses1, pg_losses2)
    pg_loss = verl_F.masked_mean(pg_losses_clip, response_mask)

    metrics = {"pg_loss": pg_loss.detach().item()}
    return pg_loss, metrics


loss, metrics = ppo_loss_fn(**args)
print(loss)
print(metrics)

In [ ]:
wg.set_actor_loss_fn(ppo_loss_fn)
actor_output = wg.train_actor(gen_out)

In [ ]:
# put it all together
import wandb

batch_size = 128
num_bon = 8
wandb.init(project="alphaseed_example", config={"batch_size": batch_size, "num_bon": num_bon})


train_dataloader = DataLoader(dataset=train_dataset,
                              batch_size=batch_size,
                              shuffle=None,
                              drop_last=True,
                              collate_fn=collate_fn)

step = 1
for batch_dict in train_dataloader:
    batch: DataProto = DataProto.from_single_dict(batch_dict, meta_info={'generation_kwargs': generation_kwargs})
    batch = set_default_values(batch, remote_config.data.max_response_length)
    prompts = tokenizer.batch_decode(batch.batch["input_ids"], skip_special_tokens=True)

    print(f"step: {step}, gen")
    batch = batch.repeat(num_bon)
    gen_out = wg.generate_sequences(batch)

    print(f"step: {step}, logprob")
    gen_out.batch["prompts"] = gen_out.batch["input_ids"][:, :max_prompt_length]
    gen_out.batch["responses"] = gen_out.batch["input_ids"][:, max_prompt_length:]
    gen_out.meta_info = {'generation_kwargs': generation_kwargs}
    gen_out = wg.old_log_probs(gen_out)

    print(f"step: {step}, reward")
    sequences = tokenizer.batch_decode(gen_out.batch["input_ids"], skip_special_tokens=True)
    response_ids = gen_out.batch['input_ids'][:, max_prompt_length:]
    reward_tensor = torch.zeros_like(response_ids, dtype=torch.float32)
    scores = []
    seqlens = []
    for i in range(batch_size):
        # compute reward
        ground_truth = batch[i].non_tensor_batch['reward_model']['ground_truth']
        score = math_v1.compute_score(sequences[i], ground_truth)
        scores.append(score)
        prompt_length = batch[i].batch['input_ids'].shape[-1]
        valid_response_length = gen_out[i].batch['attention_mask'][prompt_length:].sum().item()
        seqlens.append(valid_response_length)
        # give score to the eos token
        reward_tensor[i, valid_response_length - 1] = score

    avg_score = sum(scores)/len(scores)
    avg_seqlen = sum(seqlens)/len(seqlens)
    print(f"step: {step}, score: {avg_score}, seqlen: {avg_seqlen}")
    wandb.log({
        "train/score": avg_score,
        "train/response_length": avg_seqlen,
    }, step=step)

    print(f"step: {step}, advantage")
    response_length = gen_out.batch['responses'].size(1)
    response_mask = gen_out.batch['attention_mask'][:, -response_length:]
    index = batch.non_tensor_batch['index']
    advantages, returns, adv_metrics = core_algos.compute_grpo_advantage_return(
        token_level_scores=reward_tensor,
        eos_mask=response_mask,
        index=index,
        num_bon=num_bon)
    gen_out.batch['advantages'] = advantages
    gen_out.batch['returns'] = returns
    gen_out.batch['upgo_advantages'] = torch.zeros_like(advantages)
    gen_out.meta_info['global_token_num'] = torch.sum(gen_out.batch['attention_mask'], dim=-1).tolist()

    print(f"step: {step}, train actor")
    actor_output = wg.train_actor(gen_out)
    step += 1

wandb: ⭐️ View project at https://ml.bytedance.net/experiment/tracking/detail?Id=project_20250222_622d7b3d
wandb: 🚀 View run at https://ml.bytedance.net/experiment/tracking/detail?Id=project_20250222_622d7b3d&selectedTrial=run_20250311_3ea30208
